# CS180 Project 4A: Image Warping and Mosaicing

**Name:** TODO

Spec: https://cal-cs180.github.io/fa26/hw/proj3/partA.html

Workflow:
1. Put your photos in `data/` and fill in the config cells (`RECTIFY`, `MOSAICS`).
2. Click correspondences with `pick_points(...)` (opens a separate window; saves to `points/`).
3. Implement every function in the **Student implementation** cells (search for `TODO`).
4. Run the provided cells; results are saved to `output/`.

**Rules:** no `cv2.findHomography`, `cv2.warpPerspective`, `cv2.getPerspectiveTransform`,
`skimage.transform.ProjectiveTransform`, `skimage.transform.warp`, `scipy.interpolate.griddata`, etc.
Linear solvers / matrix inverse are fine.

**Conventions used throughout:** points are `(x, y) = (column, row)`; pixel `(r, c)` has its center at
`(x, y) = (c, r)`; images are float64 in `[0, 1]`, shape `(H, W, 3)`.

## Setup

In [ ]:
import json, subprocess, sys, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy.ndimage import gaussian_filter

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
np.set_printoptions(precision=4, suppress=True)

DATA, POINTS, OUT = Path('data'), Path('points'), Path('output')
for d in (DATA, POINTS, OUT):
    d.mkdir(exist_ok=True)

MAX_DIM = 1600  # images are downscaled so the longest side is <= MAX_DIM (keeps warps fast)

### Provided helpers

In [ ]:
def load_image(path, max_dim=MAX_DIM):
    """Load an image as float64 RGB in [0, 1], honoring EXIF rotation and downscaling."""
    im = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    if max_dim and max(im.size) > max_dim:
        s = max_dim / max(im.size)
        im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
    return np.asarray(im, dtype=np.float64) / 255.0


def save_image(im, name):
    """Save a float image in [0, 1] (RGB or grayscale) to output/<name>."""
    arr = (np.clip(im, 0, 1) * 255).round().astype(np.uint8)
    Image.fromarray(arr).save(OUT / name)
    return OUT / name


def show_images(images, titles=None, ncols=None, size=5, save=None, suptitle=None):
    """Display a list of images in a grid. `save` writes the whole figure to output/<save>."""
    n = len(images)
    ncols = ncols or n
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for i, im in enumerate(images):
        ax = axes.flat[i]
        ax.imshow(np.clip(im, 0, 1), cmap='gray' if im.ndim == 2 else None)
        if titles:
            ax.set_title(titles[i])
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    if save:
        fig.savefig(OUT / save, bbox_inches='tight', dpi=150)
    plt.show()


def show_crops(images, titles, box, size=5, save=None):
    """Show the same crop box=(x0, y0, x1, y1) from several images, for zoomed-in comparisons."""
    x0, y0, x1, y1 = box
    show_images([im[y0:y1, x0:x1] for im in images], titles, size=size, save=save)


def pick_points(name, *image_paths):
    """Open a GUI window to click correspondences; saves to points/<name>.json.

    Re-running resumes from the existing file. Close the window when done.
    """
    out = POINTS / f'{name}.json'
    subprocess.run([sys.executable, 'pick_points.py', str(out), *map(str, image_paths),
                    '--max-dim', str(MAX_DIM)], check=True)
    return load_points(name)


def load_points(name, shapes=None):
    """Load points/<name>.json -> list of (n, 2) arrays, one per image.

    If `shapes` (list of image .shape) is given and differs from the shapes the points
    were clicked on, the points are rescaled to match.
    """
    d = json.loads((POINTS / f'{name}.json').read_text())
    pts = [np.array(p, dtype=np.float64).reshape(-1, 2) for p in d['points']]
    if shapes is not None:
        for i, (old, new) in enumerate(zip(d['shapes'], shapes)):
            pts[i] = pts[i] * np.array([new[1] / old[1], new[0] / old[0]])
    return pts


def save_points(name, image_paths, pts_list, shapes):
    """Save hand-typed points (e.g. from pixspy) in the same format pick_points uses."""
    (POINTS / f'{name}.json').write_text(json.dumps({
        'images': [str(p) for p in image_paths],
        'shapes': [list(s[:2]) for s in shapes],
        'points': [np.asarray(p).tolist() for p in pts_list],
    }, indent=2))


def plot_correspondences(images, pts_list, titles=None, size=7, save=None):
    """Plot numbered points on each image; point i should be the same feature everywhere."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(size * n, size), squeeze=False)
    colors = plt.cm.tab10.colors
    for k, (ax, im, pts) in enumerate(zip(axes[0], images, pts_list)):
        ax.imshow(np.clip(im, 0, 1))
        for i, (x, y) in enumerate(pts):
            c = colors[i % len(colors)]
            ax.plot(x, y, 'o', mfc='none', mec=c, ms=10, mew=2)
            ax.text(x + 8, y - 8, str(i), color=c, fontsize=11, weight='bold')
        ax.set_title(titles[k] if titles else f'{len(pts)} points')
        ax.axis('off')
    plt.tight_layout()
    if save:
        fig.savefig(OUT / save, bbox_inches='tight', dpi=150)
    plt.show()


def apply_homography(H, pts):
    """Map (n, 2) points through H, with the homogeneous divide."""
    p = np.c_[pts, np.ones(len(pts))] @ H.T
    return p[:, :2] / p[:, 2:3]


def reprojection_error(H, pts1, pts2):
    """Mean pixel distance between H(pts1) and pts2."""
    return np.linalg.norm(apply_homography(H, pts1) - pts2, axis=1).mean()


def timed(fn, *args, **kwargs):
    """Run fn and return (result, seconds)."""
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, time.perf_counter() - t0

# A.1: Shoot the Pictures

Put your photos in `data/` and list them here. You need at least two image sets related by a projective
transform (fixed center of projection, rotate the camera, ~40–70% overlap), plus images of planar
rectangular objects for rectification.

In [ ]:
# TODO: replace with your filenames. Order mosaic images left -> right; REF is the image the others warp into.
MOSAICS = {
    'mosaic1': dict(paths=[DATA / 'm1_left.jpg', DATA / 'm1_center.jpg', DATA / 'm1_right.jpg'], ref=1),
    'mosaic2': dict(paths=[DATA / 'm2_left.jpg', DATA / 'm2_right.jpg'], ref=0),
    'mosaic3': dict(paths=[DATA / 'm3_left.jpg', DATA / 'm3_right.jpg'], ref=0),
}

# size = (width, height) of the rectified output rectangle in pixels; pick the real-world aspect ratio.
RECTIFY = {
    'rect1': dict(path=DATA / 'rect1.jpg', size=(400, 400)),
    'rect2': dict(path=DATA / 'rect2.jpg', size=(600, 400)),
}

In [ ]:
mosaic_images = {name: [load_image(p) for p in cfg['paths']] for name, cfg in MOSAICS.items()}
for name, ims in mosaic_images.items():
    show_images(ims, [p.name for p in MOSAICS[name]['paths']], suptitle=name, save=f'{name}_sources.jpg')

rect_images = {name: load_image(cfg['path']) for name, cfg in RECTIFY.items()}
show_images(list(rect_images.values()), list(rect_images.keys()), suptitle='Rectification inputs')

**A.1 write-up:** TODO — describe where/how you shot each set (camera, rotation about the center of
projection, exposure/focus locking, overlap), and where any outside images came from.

# A.2: Recover Homographies

Implement `build_homography_system` and `computeH`. `H` maps `im1_pts` to `im2_pts`
(`p' ~ H p`), with `H[2, 2] = 1`, solved by least squares from n > 4 correspondences.

### Student implementation

In [ ]:
def build_homography_system(im1_pts, im2_pts):
    """Build the linear system A h = b for the 8 unknowns of H.

    im1_pts, im2_pts: (n, 2) arrays of (x, y) correspondences.
    Returns A with shape (2n, 8) and b with shape (2n,), where
    h = [h11, h12, h13, h21, h22, h23, h31, h32] and h33 = 1.
    """
    # TODO
    raise NotImplementedError


def computeH(im1_pts, im2_pts):
    """Return the 3x3 homography H (H[2, 2] == 1) mapping im1_pts -> im2_pts via least squares."""
    # TODO: use build_homography_system + np.linalg.lstsq
    raise NotImplementedError

### Sanity check

In [ ]:
rng = np.random.default_rng(0)
H_true = np.array([[1.1, 0.05, 30.0], [-0.03, 0.95, -12.0], [1e-4, -2e-4, 1.0]])
p1 = rng.uniform(0, 800, (10, 2))
p2 = apply_homography(H_true, p1)
assert build_homography_system(p1, p2)[0].shape == (20, 8)
assert np.allclose(computeH(p1, p2), H_true, atol=1e-6), 'computeH does not recover a known H'
H_noisy = computeH(p1, p2 + rng.normal(0, 1, p2.shape))
print('OK. Reprojection error with 1px noise:', reprojection_error(H_noisy, p1, p2))

### Click correspondences

Run once per adjacent pair in each mosaic (a GUI window opens; click the same feature in both images
in the same order, then close the window). Use well-spread, sharp features (corners), at least ~8 per pair.
Re-running resumes from the saved file. Comment the line back out once you're done.

In [ ]:
def pair_name(name, i, j):
    return f'{name}_{i}_{j}'

# TODO: uncomment one pair at a time, click, then re-comment.
# pick_points(pair_name('mosaic1', 0, 1), *MOSAICS['mosaic1']['paths'][0:2])
# pick_points(pair_name('mosaic1', 1, 2), *MOSAICS['mosaic1']['paths'][1:3])
# pick_points(pair_name('mosaic2', 0, 1), *MOSAICS['mosaic2']['paths'][0:2])
# pick_points(pair_name('mosaic3', 0, 1), *MOSAICS['mosaic3']['paths'][0:2])

### Correspondences, system of equations, and recovered H

In [ ]:
pair_H = {}  # (name, i, j) -> H mapping image i -> image j
for name, ims in mosaic_images.items():
    for i in range(len(ims) - 1):
        pn = pair_name(name, i, i + 1)
        if not (POINTS / f'{pn}.json').exists():
            print(f'Skipping {pn}: no points yet')
            continue
        pts_i, pts_j = load_points(pn, [ims[i].shape, ims[i + 1].shape])
        plot_correspondences([ims[i], ims[i + 1]], [pts_i, pts_j],
                             [f'{name} [{i}]', f'{name} [{i + 1}]'], save=f'{pn}_points.jpg')
        H = computeH(pts_i, pts_j)
        pair_H[(name, i, i + 1)] = H
        print(f'{pn}: {len(pts_i)} correspondences, '
              f'reprojection error = {reprojection_error(H, pts_i, pts_j):.2f} px')
        print('H =\n', H, '\n')

# Show the system for one pair (first two correspondences = first 4 rows).
if pair_H:
    (name, i, j) = next(iter(pair_H))
    pts_i, pts_j = load_points(pair_name(name, i, j), [mosaic_images[name][i].shape, mosaic_images[name][j].shape])
    A, b = build_homography_system(pts_i, pts_j)
    print(f'A ({A.shape}), first 4 rows:\n', A[:4], '\nb, first 4 entries:\n', b[:4])

**A.2 write-up:** TODO — derive the system of equations. Starting from

$$
\begin{bmatrix} w x' \\ w y' \\ w \end{bmatrix} =
\begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & 1 \end{bmatrix}
\begin{bmatrix} x \\ y \\ 1 \end{bmatrix}
$$

show the two rows each correspondence contributes to $A\mathbf{h} = \mathbf{b}$, why $n \ge 4$ is needed,
and why you use least squares with more points. Comment on the recovered H and reprojection errors.

# A.3: Warp the Images

Inverse warping: for each output pixel, map back through $H^{-1}$ and sample the source image.

**Bounds convention:** `bounds = (xmin, ymin, xmax, ymax)` (ints) in the destination frame. The output has
shape `(ymax - ymin, xmax - xmin, C)` and output pixel `(r, c)` corresponds to destination point
`(xmin + c, ymin + r)`. Both warp functions return `(imwarped, alpha)` where `alpha` is 1 where the
source was sampled and 0 outside it. Images may have any number of channels (useful for warping a
weight mask along with the image in A.4).

### Student implementation

In [ ]:
def compute_warp_bounds(im_shape, H):
    """Return integer (xmin, ymin, xmax, ymax) covering the image's four corners mapped through H."""
    # TODO
    raise NotImplementedError


def warpImageNearestNeighbor(im, H, bounds=None):
    """Inverse-warp `im` by H with nearest-neighbor sampling.

    bounds: output region in the destination frame (see convention above);
            if None, use compute_warp_bounds(im.shape, H).
    Returns (imwarped, alpha).
    """
    # TODO
    raise NotImplementedError


def warpImageBilinear(im, H, bounds=None):
    """Inverse-warp `im` by H with bilinear interpolation (weighted average of 4 neighbors).

    Same interface as warpImageNearestNeighbor. Returns (imwarped, alpha).
    """
    # TODO
    raise NotImplementedError

### Sanity check

In [ ]:
_test = np.random.default_rng(1).random((40, 60, 3))
_T = np.array([[1, 0, 5], [0, 1, -3], [0, 0, 1]], dtype=float)  # integer translation
for warp in (warpImageNearestNeighbor, warpImageBilinear):
    out, alpha = warp(_test, np.eye(3))
    assert out.shape == _test.shape and np.allclose(out, _test) and alpha.min() == 1, f'{warp.__name__}: identity failed'
    out, alpha = warp(_test, _T, bounds=(5, -3, 65, 37))
    assert np.allclose(out, _test), f'{warp.__name__}: translation failed'
    assert compute_warp_bounds(_test.shape, _T)[:2] == (5, -3)
print('OK')

### Rectification

Click the 4 corners of the known rectangle in each image in the order
**top-left, top-right, bottom-right, bottom-left**. They are mapped to a `width x height` rectangle.
Set `CROP = True` to only keep the rectified rectangle; `False` warps the whole image (can get large if
the image contains the horizon).

In [ ]:
# TODO: uncomment, click 4 corners (TL, TR, BR, BL), close the window, then re-comment.
# pick_points('rect1', RECTIFY['rect1']['path'])
# pick_points('rect2', RECTIFY['rect2']['path'])

In [ ]:
CROP = False
PAD = 0.25  # when CROP, keep this fraction of the rectangle size as context around it

rectified = {}
for name, cfg in RECTIFY.items():
    im = rect_images[name]
    (corners,) = load_points(name, [im.shape])
    assert len(corners) == 4, f'{name}: need exactly 4 corners, got {len(corners)}'
    w, h = cfg['size']
    target = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=float)
    H = computeH(corners, target)
    bounds = (int(-PAD * w), int(-PAD * h), int((1 + PAD) * w), int((1 + PAD) * h)) if CROP else None

    (nn, _), t_nn = timed(warpImageNearestNeighbor, im, H, bounds)
    (bil, _), t_bil = timed(warpImageBilinear, im, H, bounds)
    rectified[name] = dict(nn=nn, bil=bil, t_nn=t_nn, t_bil=t_bil)

    plot_correspondences([im], [corners], [f'{name}: selected corners'])
    show_images([nn, bil], [f'Nearest neighbor ({t_nn:.2f}s)', f'Bilinear ({t_bil:.2f}s)'],
                suptitle=f'{name} rectified', save=f'{name}_rectified_compare.jpg')
    save_image(nn, f'{name}_rectified_nn.jpg')
    save_image(bil, f'{name}_rectified_bilinear.jpg')

### Interpolation comparison (zoomed in)

In [ ]:
# TODO: pick a crop box (x0, y0, x1, y1) on an edge/texture where the difference is visible.
ZOOM = {name: (0, 0, 150, 150) for name in RECTIFY}

for name, r in rectified.items():
    show_crops([r['nn'], r['bil']], ['Nearest neighbor', 'Bilinear'], ZOOM[name],
               save=f'{name}_interp_zoom.jpg')
    print(f"{name}: NN {r['t_nn']:.3f}s, bilinear {r['t_bil']:.3f}s "
          f"({r['t_bil'] / r['t_nn']:.1f}x slower)")

**A.3 write-up:** TODO — explain your inverse-warping implementation (bounds, `H^{-1}`, the
pixel-center convention, how alpha is computed). Discuss the rectification results and compare
nearest neighbor vs. bilinear: visual quality (aliasing/jaggies vs. smoothing) and speed, using the
timings above.

# A.4: Blend the Images into a Mosaic

Each image is warped into the reference image's frame. `Hs[i]` maps image `i` to the reference
(`Hs[ref]` is the identity); these are chained from the pairwise homographies in A.2.

### Student implementation

In [ ]:
def make_alpha_mask(h, w):
    """Feathering weights for an h x w image: 1 at the center, falling linearly to 0 at the edges."""
    # TODO
    raise NotImplementedError


def build_mosaic(images, Hs, warp_fn=None):
    """Warp every image into a common frame and blend with weighted averaging (feathering).

    images: list of (h, w, 3) images.
    Hs: list of 3x3 homographies; Hs[i] maps image i into the reference frame.
    warp_fn: warpImageBilinear or warpImageNearestNeighbor.
    Returns the mosaic, shape (H, W, 3).

    Suggested steps:
      1. Union of compute_warp_bounds over all images -> one shared output bounds.
      2. Stack make_alpha_mask onto each image as a 4th channel and warp it with the shared bounds.
      3. Blend with the warped weights (simple weighted average, or a 2-level Laplacian pyramid
         using gaussian_filter to reduce ghosting).
    """
    warp_fn = warp_fn or warpImageBilinear
    # TODO
    raise NotImplementedError

### Chain homographies to the reference image

In [ ]:
def homographies_to_ref(name):
    """Hs[i]: image i -> reference, built from the adjacent-pair homographies in pair_H."""
    n, ref = len(mosaic_images[name]), MOSAICS[name]['ref']
    Hs = [None] * n
    Hs[ref] = np.eye(3)
    for i in range(ref - 1, -1, -1):  # left of ref: i -> i+1 -> ... -> ref
        Hs[i] = Hs[i + 1] @ pair_H[(name, i, i + 1)]
    for i in range(ref + 1, n):       # right of ref: i -> i-1 -> ... -> ref
        Hs[i] = Hs[i - 1] @ np.linalg.inv(pair_H[(name, i - 1, i)])
    return [H / H[2, 2] for H in Hs]

In [ ]:
mosaics = {}
for name, ims in mosaic_images.items():
    try:
        Hs = homographies_to_ref(name)
    except KeyError as e:
        print(f'Skipping {name}: missing correspondences for pair {e}')
        continue
    mosaic, secs = timed(build_mosaic, ims, Hs)
    mosaics[name] = mosaic
    show_images(ims, [p.name for p in MOSAICS[name]['paths']], suptitle=f'{name}: sources')
    show_images([mosaic], [f'{name} ({secs:.1f}s)'], size=14)
    save_image(mosaic, f'{name}.jpg')

**A.4 write-up:** TODO — explain your procedure: choice of reference/projection, how the output canvas
is sized, warping, the alpha masks, and the blending method. Show/discuss any artifacts (ghosting, seams,
exposure differences) and what you did about them.

**Mosaic 1:** TODO

**Mosaic 2:** TODO

**Mosaic 3:** TODO